In [ ]:
import os
import json
import datetime
import uuid
from xml.sax.saxutils import escape
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak, Table, TableStyle
from reportlab.lib import colors
from config import *

In [ ]:
def generate_session_id(crime_type):
    code = CRIME_TYPE_CODES.get(crime_type, "xx")
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    unique_suffix = uuid.uuid4().hex[:8]
    return f"{code}_{timestamp}_{unique_suffix}"

In [ ]:
def save_session(scenario, checklist, history, coverage_flags, narrative, narrative_coverage, grading_results):
    session_id = generate_session_id(scenario["crime_type"])
    os.makedirs(SESSIONS_DIR, exist_ok=True)

    # Save JSON
    session_data = {
        "session_id": session_id,
        "timestamp": datetime.datetime.now().isoformat(),
        "metadata": {
            "crime_type": scenario["crime_type"],
            "interviewee_name": scenario["interviewee_name"],
            "interviewee_role": scenario["interviewee_role"],
            "dispatch_line": scenario["dispatch_line"]
        },
        "scenario": scenario,
        "interview_transcript": history,
        "coverage_flags": coverage_flags,
        "narrative": narrative,
        "narrative_coverage": narrative_coverage,
        "grading_results": grading_results
    }

    json_path = os.path.join(SESSIONS_DIR, f"{session_id}.json")
    with open(json_path, "w") as f:
        json.dump(session_data, f, indent=2)
    print(f"Session saved: {json_path}")

    # Generate and save PDF
    pdf_path = generate_pdf_report(session_id, scenario, checklist, grading_results)

    return session_id, json_path, pdf_path

In [ ]:
def generate_pdf_report(session_id, scenario, checklist, grading_results):
    pdf_path = os.path.join(SESSIONS_DIR, f"{session_id}.pdf")

    doc = SimpleDocTemplate(
        pdf_path,
        pagesize=letter,
        leftMargin=0.75*inch,
        rightMargin=0.75*inch,
        topMargin=0.75*inch,
        bottomMargin=0.75*inch
    )

    styles = getSampleStyleSheet()

    title_style = ParagraphStyle(
        'Title',
        parent=styles['Heading1'],
        fontSize=16,
        spaceAfter=6
    )
    heading_style = ParagraphStyle(
        'Heading',
        parent=styles['Heading2'],
        fontSize=13,
        spaceAfter=6,
        spaceBefore=12
    )
    normal_style = ParagraphStyle(
        'Normal',
        parent=styles['Normal'],
        fontSize=9,
        spaceAfter=4,
        leading=13
    )
    small_style = ParagraphStyle(
        'Small',
        parent=styles['Normal'],
        fontSize=8,
        spaceAfter=3,
        textColor=colors.HexColor('#555555'),
        leading=11
    )

    table_cell_style = ParagraphStyle(
        'TableCell',
        parent=styles['Normal'],
        fontSize=7,
        leading=9,
        wordWrap='CJK'
    )

    table_header_style = ParagraphStyle(
        'TableHeader',
        parent=table_cell_style,
        textColor=colors.white
    )

    def table_cell(value, style=table_cell_style):
        if value is None:
            value = "—"
        text = escape(str(value)).replace("\n", "<br/>")
        return Paragraph(text, style)

    def truncate(value, max_len=60):
        value = "" if value is None else str(value)
        return value[:max_len] + ("..." if len(value) > max_len else "")

    def format_limited_refs(items, formatter, max_items=3):
        items = [
            (key, value)
            for key, value in items.items()
            if value and str(value).strip()
        ]

        shown_items = items[:max_items]

        refs = [
            formatter(key, value)
            for key, value in shown_items
        ]

        remaining = len(items) - len(shown_items)
        if remaining > 0:
            refs.append(f"... and {remaining} more")

        return "\n".join(refs) if refs else "—"

    story = []

    # ── Summary header ────────────────────────────────────────────────
    story.append(Paragraph("Police Report Training — Grading Report", title_style))
    story.append(Paragraph(f"Session: {session_id}", small_style))
    story.append(Paragraph(f"Crime type: {scenario['crime_type']}", normal_style))
    story.append(Paragraph(f"Interviewee: {scenario['interviewee_name']} ({scenario['interviewee_role']})", normal_style))
    story.append(Paragraph(f"Dispatch: {scenario['dispatch_line']}", normal_style))
    story.append(Spacer(1, 8))

    content = grading_results["content"]
    total = len(content)
    full_interview = sum(1 for r in content if r["asked_in_interview"] == "full")
    partial_interview = sum(1 for r in content if r["asked_in_interview"] == "partial")
    full_narrative = sum(1 for r in content if r["in_narrative"] == "full")
    partial_narrative = sum(1 for r in content if r["in_narrative"] == "partial")
    contradictions = len(grading_results["contradictions_interview"]) + len(grading_results["contradictions_narrative"])
    grammar_errors = len(grading_results["grammar"])
    style_errors = len(grading_results["style"])

    summary_data = [
        ["Metric", "Result"],
        ["Total checklist elements", str(total)],
        ["Fully asked in interview", str(full_interview)],
        ["Partially asked in interview", str(partial_interview)],
        ["Fully covered in narrative", str(full_narrative)],
        ["Partially covered in narrative", str(partial_narrative)],
        ["Contradictions found", str(contradictions)],
        ["Grammar errors", str(grammar_errors)],
        ["Style violations", str(style_errors)],
    ]

    summary_table = Table(summary_data, colWidths=[3.5*inch, 2*inch])
    summary_table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#1a1a2e')),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('FONTSIZE', (0, 0), (-1, -1), 9),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor('#f5f5f5')]),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor('#cccccc')),
        ('PADDING', (0, 0), (-1, -1), 5),
    ]))
    story.append(summary_table)
    story.append(Spacer(1, 12))

    # ── Section 1 — Content coverage table ───────────────────────────
    story.append(Paragraph("Section 1 — Content coverage", heading_style))

    content_data = [[
        table_cell("Element ID", table_header_style),
        table_cell("Element", table_header_style),
        table_cell("Asked in interview", table_header_style),
        table_cell("Interview context", table_header_style),
        table_cell("In narrative", table_header_style),
        table_cell("Narrative ref", table_header_style),
        table_cell("Notes", table_header_style)
    ]]

    for row in content:
        interview_status = row["asked_in_interview"]
        interview_source_label = row.get("interview_source_label")

        if interview_status == "full":
            interview_label = "Full"
            if interview_source_label:
                interview_label += f" — {interview_source_label}"
        elif interview_status == "partial":
            interview_label = "Partial"
            if interview_source_label:
                interview_label += f" — {interview_source_label}"
        else:
            interview_label = "Not asked"

        narrative_status = row["in_narrative"]
        if narrative_status == "full":
            narrative_label = "Full"
        elif narrative_status == "partial":
            narrative_label = "Partial"
        else:
            narrative_label = "Missing"

        # Interview context — line numbers and messages
        if interview_status in ("full", "partial") and row["interview_messages"]:
            if row.get("interview_source") == "interviewee_response":
                interview_ref = "\n".join([
                    f"Line {line} response: \"{truncate(msg.get('provided_fact') or msg.get('interviewee_response', ''))}\""
                    for line, msg in row["interview_messages"].items()
                ])
            else:
                interview_ref = "\n".join([
                    f"Line {line} question: \"{truncate(msg.get('officer_message', ''))}\""
                    for line, msg in row["interview_messages"].items()
                ])
        else:
            interview_ref = "—"

        # Narrative ref — chunk numbers and phrases
        if narrative_status in ("full", "partial") and row["narrative_phrases"]:
            narrative_ref = "\n".join([
                f"Chunk {chunk}: \"{phrase[:60]}{'...' if len(phrase) > 60 else ''}\""
                for chunk, phrase in row["narrative_phrases"].items()
            ])
        else:
            narrative_ref = "—"

        notes = row["notes"] if row["notes"] else "—"

        content_data.append([
            table_cell(row["element_id"]),
            table_cell(row["element"]),
            table_cell(interview_label),
            table_cell(interview_ref),
            table_cell(narrative_label),
            table_cell(narrative_ref),
            table_cell(notes)
        ])

    content_table = Table(content_data, colWidths=[0.6*inch, 1.2*inch, 0.75*inch, 1.2*inch, 0.65*inch, 1.2*inch, 1.2*inch])

    content_table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#1a1a2e')),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('FONTSIZE', (0, 0), (-1, -1), 7),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor('#f5f5f5')]),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor('#cccccc')),
        ('PADDING', (0, 0), (-1, -1), 4),
        ('VALIGN', (0, 0), (-1, -1), 'TOP'),
    ]))
    story.append(content_table)
    story.append(PageBreak())

    # ── Section 2 — Grammar errors ───────────────────────────────────
    story.append(Paragraph("Section 2 — Grammar errors", heading_style))

    if grading_results["grammar"]:
        grammar_data = [[
            table_cell("Chunk", table_header_style),
            table_cell("Source", table_header_style),
            table_cell("Error type", table_header_style),
            table_cell("Excerpt", table_header_style),
            table_cell("Suggestion", table_header_style)
        ]]
        for err in grading_results["grammar"]:
            grammar_data.append([
                table_cell(str(err.get("chunk", "—"))),
                table_cell(err.get("source", "—")),
                table_cell(err.get("error_type", "—")),
                table_cell(err.get("excerpt", "—")),
                table_cell(err.get("suggestion", "—"))
            ])
        grammar_table = Table(grammar_data, colWidths=[0.5*inch, 0.7*inch, 1*inch, 2.3*inch, 2.3*inch])
        grammar_table.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#1a1a2e')),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
            ('FONTSIZE', (0, 0), (-1, -1), 7),
            ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor('#f5f5f5')]),
            ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor('#cccccc')),
            ('PADDING', (0, 0), (-1, -1), 4),
            ('VALIGN', (0, 0), (-1, -1), 'TOP'),
        ]))
        story.append(grammar_table)
    else:
        story.append(Paragraph("No grammar errors found.", normal_style))

    story.append(PageBreak())

    # ── Section 3 — Style violations ─────────────────────────────────
    story.append(Paragraph("Section 3 — Style violations", heading_style))

    if grading_results["style"]:
        style_data = [[
            table_cell("Chunk", table_header_style),
            table_cell("Rule", table_header_style),
            table_cell("Error type", table_header_style),
            table_cell("Excerpt", table_header_style),
            table_cell("Suggestion", table_header_style)
        ]]
        for err in grading_results["style"]:
            style_data.append([
                table_cell(str(err.get("chunk", "—"))),
                table_cell(err.get("rule", "—")),
                table_cell(err.get("error_type", "—")),
                table_cell(err.get("excerpt", "—")),
                table_cell(err.get("suggestion", "—"))
            ])
        style_table = Table(style_data, colWidths=[0.5*inch, 0.7*inch, 1*inch, 2.3*inch, 2.3*inch])
        style_table.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#1a1a2e')),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
            ('FONTSIZE', (0, 0), (-1, -1), 7),
            ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor('#f5f5f5')]),
            ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor('#cccccc')),
            ('PADDING', (0, 0), (-1, -1), 4),
            ('VALIGN', (0, 0), (-1, -1), 'TOP'),
        ]))
        story.append(style_table)
    else:
        story.append(Paragraph("No style violations found.", normal_style))

    doc.build(story)
    print(f"PDF saved: {pdf_path}")
    return pdf_path